# Home Credit Default Risk EDA

This notebook is designed for portfolio presentation. It focuses on:

- business framing of the credit-risk problem
- target imbalance and why ROC-AUC / recall matter
- borrower-level patterns in income, credit amount, annuity, and external risk scores
- signals from prior applications and bureau history
- insights that support downstream modeling decisions


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import HOME_CREDIT_DIR

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 200)


In [ ]:
application = pd.read_csv(HOME_CREDIT_DIR / 'application_train.csv')
previous = pd.read_csv(HOME_CREDIT_DIR / 'previous_application.csv')
bureau = pd.read_csv(HOME_CREDIT_DIR / 'bureau.csv')

print('application_train shape:', application.shape)
print('previous_application shape:', previous.shape)
print('bureau shape:', bureau.shape)


## 1. Target Imbalance

The dataset is highly imbalanced. That means raw accuracy can be misleading, so recall, precision, F1, and ROC-AUC matter more.

In [ ]:
target_rate = application['TARGET'].value_counts(normalize=True).sort_index()
display(target_rate)

ax = target_rate.rename({0: 'No Default', 1: 'Default'}).plot(kind='bar', figsize=(6, 4), color=['#2a9d8f', '#e76f51'])
ax.set_title('Class Distribution')
ax.set_ylabel('Share of applicants')
plt.xticks(rotation=0)
plt.show()


## 2. Missing Values

The Home Credit data contains substantial missingness. Understanding which fields are sparse helps justify imputation choices.

In [ ]:
missing = (application.isna().mean() * 100).sort_values(ascending=False)
missing.head(20).to_frame('missing_pct')


## 3. Income, Credit, and Annuity

These variables help tell the borrower affordability story.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.boxplot(data=application, x='TARGET', y='AMT_INCOME_TOTAL', ax=axes[0])
axes[0].set_title('Income by Default')
axes[0].set_ylim(0, application['AMT_INCOME_TOTAL'].quantile(0.95))

sns.boxplot(data=application, x='TARGET', y='AMT_CREDIT', ax=axes[1])
axes[1].set_title('Credit Amount by Default')
axes[1].set_ylim(0, application['AMT_CREDIT'].quantile(0.95))

sns.boxplot(data=application, x='TARGET', y='AMT_ANNUITY', ax=axes[2])
axes[2].set_title('Annuity by Default')
axes[2].set_ylim(0, application['AMT_ANNUITY'].quantile(0.95))
plt.tight_layout()
plt.show()


## 4. External Risk Scores

The `EXT_SOURCE` features are usually among the strongest predictive signals in this competition.

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ext_cols):
    sns.kdeplot(data=application, x=col, hue='TARGET', common_norm=False, ax=ax)
    ax.set_title(f'{col} distribution by default')
plt.tight_layout()
plt.show()


## 5. Previous Applications

Borrowers with more prior applications may behave differently from first-time applicants.

In [ ]:
prev_counts = previous.groupby('SK_ID_CURR').size().rename('PREV_APPLICATION_COUNT').reset_index()
application_prev = application[['SK_ID_CURR', 'TARGET']].merge(prev_counts, on='SK_ID_CURR', how='left')
application_prev['PREV_APPLICATION_COUNT'] = application_prev['PREV_APPLICATION_COUNT'].fillna(0)

sns.boxplot(data=application_prev, x='TARGET', y='PREV_APPLICATION_COUNT')
plt.title('Previous Application Count by Default')
plt.show()


## 6. Bureau History

External bureau history helps show whether the borrower already has active or overdue credit relationships.

In [ ]:
bureau_counts = bureau.groupby('SK_ID_CURR').size().rename('BUREAU_RECORD_COUNT').reset_index()
application_bureau = application[['SK_ID_CURR', 'TARGET']].merge(bureau_counts, on='SK_ID_CURR', how='left')
application_bureau['BUREAU_RECORD_COUNT'] = application_bureau['BUREAU_RECORD_COUNT'].fillna(0)

sns.boxplot(data=application_bureau, x='TARGET', y='BUREAU_RECORD_COUNT')
plt.title('Bureau Record Count by Default')
plt.show()


## 7. Key Takeaways

- The target is imbalanced, so business-aware metrics matter more than accuracy alone.
- External score features are likely to be very predictive.
- Historical tables such as bureau and previous applications carry behavior signals beyond the main application file.
- This justifies using multi-table feature engineering in the final modeling pipeline.
